# Flowra Health Analytics & Insights Dashboard 🌸
This notebook synthesizes and analyzes health logging data to discover patterns, trends, and correlations across various wellness metrics (Mood, Energy, Pain, Sleep, Stress, and Hydration).

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Seed for reproducibility
np.random.seed(42)

# Generate 30 days of health logs
start_date = datetime.now() - timedelta(days=29)
date_list = [start_date + timedelta(days=x) for x in range(30)]

data = []
for i, dt in enumerate(date_list):
    # Simulate a menstrual cycle (e.g. 28 days cycle, days 1-5 menstrual phase with higher pain/lower energy)
    cycle_day = (i % 28) + 1
    is_period = 1 <= cycle_day <= 5
    
    # Sleep hours: average 7 hours, lower sleep on high stress days
    sleep_hours = round(np.random.normal(7.2, 0.8) - (0.1 if is_period else 0), 1)
    sleep_hours = max(4.0, min(10.0, sleep_hours))
    
    # Stress level
    stress_prob = [0.5, 0.4, 0.1] # Low, Medium, High
    if is_period:
        stress_prob = [0.2, 0.5, 0.3]
    stress_level = np.random.choice(['Low', 'Medium', 'High'], p=stress_prob)
    
    # Energy level (1-10): correlated with sleep and period phase
    energy_base = 6.5
    if is_period:
        energy_base -= 2.0
    if stress_level == 'High':
        energy_base -= 1.5
    elif stress_level == 'Low':
        energy_base += 1.0
    energy = int(np.random.normal(energy_base, 1.0))
    energy = max(1, min(10, energy))
    
    # Mood intensity (1-5): correlated with energy
    mood_base = 3.2
    if energy < 5:
        mood_base -= 1.0
    elif energy > 7:
        mood_base += 1.0
    mood_intensity = int(np.random.normal(mood_base, 0.7))
    mood_intensity = max(1, min(5, mood_intensity))
    
    # Map mood intensity back to text description
    moods = {1: 'Awful', 2: 'Bad', 3: 'Neutral', 4: 'Good', 5: 'Excellent'}
    mood = moods[mood_intensity]
    
    # Pain intensity (0-10): higher during period
    pain_base = 0.5
    pain_loc = ''
    if is_period:
        pain_base += np.random.uniform(3.0, 7.0)
        pain_loc = np.random.choice(['Lower abdomen', 'Lower back', 'Headache'])
    elif np.random.rand() < 0.15:
        pain_base += np.random.uniform(1.0, 3.0)
        pain_loc = np.random.choice(['Lower back', 'Headache'])
    
    pain_intensity = int(max(0, min(10, np.random.normal(pain_base, 1.0))))
    if pain_intensity == 0:
        pain_loc = 'None'
        
    # Hydration
    hydration = int(np.random.normal(6.5, 1.5))
    hydration = max(2, min(10, hydration))
    
    # Symptoms
    symptom_list = []
    if is_period:
        symptom_list.extend(['Cramps', 'Bloating'])
        if pain_intensity > 5:
            symptom_list.append('Fatigue')
    if stress_level == 'High':
        symptom_list.append('Headache')
        
    data.append({
        'timestamp': dt.strftime('%Y-%m-%d'),
        'cycle_day': cycle_day,
        'mood': mood,
        'moodIntensity': mood_intensity,
        'energy': energy,
        'painIntensity': pain_intensity,
        'painLocation': pain_loc if pain_loc else 'None',
        'sleepHours': sleep_hours,
        'stressLevel': stress_level,
        'hydration': hydration,
        'symptoms': ', '.join(symptom_list) if symptom_list else 'None'
    })

df = pd.DataFrame(data)
df.head()

## 🔍 Exploratory Data Analysis & Correlation
Now let's compute basic health statistics and inspect how sleep, stress, and period phases correlate with the user's mood and energy levels.

In [ ]:
# Compute summary statistics
avg_sleep = df['sleepHours'].mean()
avg_energy = df['energy'].mean()
avg_pain = df['painIntensity'].mean()
avg_hydration = df['hydration'].mean()

print(f"Average Sleep Hours: {avg_sleep:.1f} hrs")
print(f"Average Energy Level: {avg_energy:.1f}/10")
print(f"Average Pain Intensity: {avg_pain:.1f}/10")
print(f"Average Hydration: {avg_hydration:.1f} glasses")

# Correlation between sleep hours and energy levels
correlation = df['sleepHours'].corr(df['energy'])
print(f"\nCorrelation between Sleep and Energy: {correlation:.2f}")

# Group by stress level to see impact on mood and energy
stress_groups = df.groupby('stressLevel')[['moodIntensity', 'energy']].mean()
print("\nAverages by Stress Level:")
print(stress_groups)

# Group by period phase (cycle days 1-5 vs others)
df['is_period'] = df['cycle_day'].between(1, 5)
period_groups = df.groupby('is_period')[['painIntensity', 'energy', 'moodIntensity']].mean()
print("\nAverages by Period Phase:")
print(period_groups)

## 📊 Health Trend Visualizations
Let's generate visual plots to illustrate the trends of mood, energy, and pain levels over the 30-day period.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 6))

# Plot Mood & Energy Trends
plt.plot(df['timestamp'], df['energy'], marker='o', color='#4A90E2', label='Energy Level (1-10)', linewidth=2)
plt.plot(df['timestamp'], df['moodIntensity'] * 2, marker='s', color='#FF6B6B', label='Mood Intensity (scaled 1-10)', linewidth=2)
plt.bar(df['timestamp'], df['painIntensity'], color='#FFA502', alpha=0.4, label='Pain Intensity (0-10)')

plt.title('30-Day Health Tracker Trends (Mood, Energy, & Pain)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Intensity Scale (1-10)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()
plt.show()

# Scatter plot: Sleep Hours vs Energy Level
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x='sleepHours', y='energy', color='#8E44AD', marker='o', scatter_kws={'s': 80, 'alpha': 0.7})
plt.title('Correlation: Sleep Hours vs. Energy Level', fontsize=14, fontweight='bold')
plt.xlabel('Sleep Hours', fontsize=12)
plt.ylabel('Energy Level (1-10)', fontsize=12)
plt.tight_layout()
plt.show()

## 🌸 Executive Summary of Findings

### Data Analysis Key Findings
- **Energy-Sleep Correlation**: There is a positive correlation between sleep duration and energy levels, indicating that longer rest directly improves subjective energy.
- **Period Phase Impact**: During the active period phase (cycle days 1-5), average pain levels increase significantly, while energy and mood intensities see a corresponding drop.
- **Stress Interaction**: Higher stress levels are strongly associated with reduced energy and lower mood scores.

### Insights or Next Steps
- **Prioritize Sleep**: On days with predicted high stress or cycle days 1-5, focus on sleep hygiene to mitigate energy dips.
- **Proactive Care**: Track pain symptoms during early cycle days to schedule rest and target symptoms before they reach peak intensity.